In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])


def find_model_path(model_name: str = 'gpt-oss-120b') -> str:
    """Auto-discover model path from /kaggle/input/."""
    import glob

    candidates = [
        # model_sources mount pattern (new Kaggle)
        f'/kaggle/input/models/danielhanchen/{model_name}/transformers/default/1',
        f'/kaggle/input/models/openai/{model_name}/transformers/default/1',
        # dataset-style mount pattern (old Kaggle / some configs)
        f'/kaggle/input/{model_name}/transformers/default/1',
        f'/kaggle/input/{model_name}',
    ]

    for path in candidates:
        if os.path.isfile(os.path.join(path, 'config.json')):
            print(f'Model found at: {path}')
            return path

    # Fallback: search for any config.json with model weights
    for config_path in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        parent = os.path.dirname(config_path)
        if any(glob.glob(os.path.join(parent, f'*.{ext}')) for ext in ['safetensors', 'bin']):
            print(f'Model found via search: {parent}')
            return parent

    # Debug dump
    print('ERROR: Model not found. /kaggle/input/ contents:')
    for root, dirs, files in os.walk('/kaggle/input/', topdown=True):
        depth = root.replace('/kaggle/input/', '').count(os.sep)
        if depth < 5:
            print(f'{"  " * depth}{os.path.basename(root)}/ ({len(files)} files)')
        if depth >= 5:
            dirs[:] = []

    raise RuntimeError(f'Model "{model_name}" not found under /kaggle/input/')


MODEL_PATH = find_model_path()
print(f'\nUsing model path: {MODEL_PATH}')

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.\n\n'
        
        '# Efficiency:\n'
        'If the problem has an obvious, immediate answer (e.g. direct computation, '
        'well-known identity, or trivial formula application), state the answer '
        'directly with brief justification. Do not use Python for problems you can '
        'solve in 3 lines of reasoning.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds\n'
        '- For very large exponents (e.g. a^(n!)), use pow(base, exp, mod) or analytical methods — never materialize the full number\n\n'
        
        '# Code Robustness Rules:\n'
        '- Each code cell must be SELF-CONTAINED: re-import and re-define everything you need. '
        'If a previous cell errored, its definitions are LOST.\n'
        '- Avoid sympy expressions that grow beyond a few hundred characters. '
        'If an expression is exploding, switch to numerical methods (numpy) or modular arithmetic.\n'
        '- For geometry: prefer numpy coordinate models with brute-force verification '
        'over pure symbolic or direction-counting approaches.\n'
        '- For combinatorics counting: always verify no duplicates — check injectivity explicitly.\n'
        '- If 5+ code cells without progress, STOP and restart with a different approach.\n'
        '- Never enumerate more than 10^6 cases without checking feasibility first.'
    )
    
    served_model_name = 'gpt-oss'
    model_path = MODEL_PATH
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 30
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 3
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 0.5
    temp_schedule = [0.3, 0.4, 0.5, 0.5, 0.6, 0.7, 0.8, 0.9]
    min_p = 0.02

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        temperature: float = None
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'Conversation': [],
                'Time': 0
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
        conversation_log = []
        turn_num = 0
        attempt_start = time.time()
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                turn_num += 1
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=(temperature if temperature is not None else self.cfg.temperature), 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                turn_text = ''.join(text_chunks)
                if turn_text.strip():
                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'reasoning',
                        'text': turn_text
                    })
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    raw_code = last_message.content[0].text
                    tool_responses = local_tool.process_sync_plus(last_message)
    
                    response_text = tool_responses[0].content[0].text
    
                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'code_call',
                        'code': raw_code,
                        'output': response_text,
                        'error': response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text
                    })
    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
    
        attempt_elapsed = time.time() - attempt_start
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': final_answer,
            'Conversation': conversation_log,
            'Time': round(attempt_elapsed, 1)
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        # Deterministic tie-breaking: score (desc), then votes (desc), then answer value (desc)
        scored_answers.sort(key=lambda x: (x['score'], x['votes'], x['answer']), reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            temp = self.cfg.temp_schedule[attempt_index % len(self.cfg.temp_schedule)]
            tasks.append((self.cfg.system_prompt, attempt_index, temp))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index, attempt_temp) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline,
                    attempt_temp
                )
    
                futures.append(future)
    
            # Collect ALL results — don't break on early stop.
            # Early stop signals remaining attempts to exit quickly,
            # but we still gather everything for a robust vote.
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    if not stop_event.is_set():
                        counts = Counter(valid_answers).most_common(1)
    
                        if counts and counts[0][1] >= self.cfg.early_stop:
                            stop_event.set()
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
    
        # --- Retry on all-None: if no valid answers and time remains, retry ---
        if not valid_answers and time.time() < deadline - 30:
            print('  >> All attempts returned None — retrying with remaining budget...')
            retry_stop = threading.Event()
            retry_executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
            try:
                retry_futures = []
                for retry_idx in range(self.cfg.attempts):
                    temp = self.cfg.temp_schedule[retry_idx % len(self.cfg.temp_schedule)]
                    # Use higher temperatures on retry to diversify
                    retry_temp = min(temp + 0.2, 1.0)
                    f = retry_executor.submit(
                        self._process_attempt,
                        user_input,
                        self.cfg.system_prompt,
                        retry_idx + self.cfg.attempts,
                        retry_stop,
                        deadline,
                        retry_temp
                    )
                    retry_futures.append(f)
                for future in as_completed(retry_futures):
                    try:
                        result = future.result()
                        detailed_results.append(result)
                        if result['Answer'] is not None:
                            valid_answers.append(result['Answer'])
                        if not retry_stop.is_set():
                            counts = Counter(valid_answers).most_common(1)
                            if counts and counts[0][1] >= self.cfg.early_stop:
                                retry_stop.set()
                    except Exception as exc:
                        print(f'Retry future failed: {exc}')
            finally:
                retry_stop.set()
                retry_executor.shutdown(wait=True, cancel_futures=True)

        self.problems_remaining = max(0, self.problems_remaining - 1)
    
        self._last_detailed_results = detailed_results
    
        if detailed_results:
            display_cols = ['Attempt', 'Answer', 'Python Calls', 'Python Errors', 'Response Length', 'Entropy', 'Time']
            results_dataframe = pd.DataFrame(detailed_results)
            display_df = results_dataframe[[c for c in display_cols if c in results_dataframe.columns]].copy()
            display_df['Entropy'] = display_df['Entropy'].round(3)
            display_df['Answer'] = display_df['Answer'].astype('Int64')
            
            display(display_df)
    
        if not valid_answers:
            print('\nResult: 0\n')
    
            return 0
    
        return self._select_answer(detailed_results)
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()

else:
    # Find test.csv — path varies between test runs and competition reruns
    test_csv_candidates = [
        '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',
        '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',
    ]
    test_csv = next((p for p in test_csv_candidates if os.path.isfile(p)), test_csv_candidates[0])
    print(f'Using test CSV: {test_csv}')

    inference_server.run_local_gateway(
        (test_csv,)
    )

In [ ]:
# =============================================================================
# ENHANCED TEST FRAMEWORK — Maximum Diagnostic Logging
# =============================================================================
# Diagnostic output is written to /kaggle/working/diagnostic.log (persists as kernel output)
# This avoids Kaggle's stdout truncation and UI hanging on large logs.

import sys

class TeeLogger:
    """Write to both stdout and a log file. Survives Kaggle log truncation."""
    def __init__(self, filepath='/kaggle/working/diagnostic.log'):
        self.terminal = sys.stdout
        self.log = open(filepath, 'w', buffering=1)  # line-buffered for live output
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = TeeLogger()
sys.stderr = TeeLogger('/kaggle/working/diagnostic_stderr.log')
# TEST LEVELS (only runs in test mode, skipped during competition re-run):
#   1 = 10 reference problems (~1-2 hours) — real difficulty
#   2 = + 10 hard diagnostic problems (~2-3 hours) — hand-picked, mostly hard
#   3 = + 10 random problems (~1-2 hours) — sampled from remaining pool
#   4 = + 30 fixed comprehensive (~5-8 hours) — full benchmark
# Total at level 4: ~50 unique problems. All layers enabled.
# =============================================================================
TEST_LEVEL = 4

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):

    def _print_attempt_full(r, expected):
        """Print complete diagnostic for one attempt — every step, no truncation."""
        attempt = r.get('Attempt', '?')
        ans = r.get('Answer')
        ent = r.get('Entropy', float('inf'))
        pcalls = r.get('Python Calls', 0)
        perrs = r.get('Python Errors', 0)
        toks = r.get('Response Length', 0)
        t = r.get('Time', 0)
        ent_s = f'{ent:.3f}' if ent != float('inf') else 'inf'
        correct_marker = ' << CORRECT' if ans == expected else (' << WRONG' if ans is not None else ' << NO ANSWER')

        print(f'\n     {"- "*35}')
        print(f'     ATTEMPT {attempt}  |  answer={ans}  entropy={ent_s}  '
              f'code_calls={pcalls}  errors={perrs}  tokens={toks}  time={t}s{correct_marker}')
        print(f'     {"- "*35}')

        # Extract libraries from all code in this attempt
        conv = r.get('Conversation', [])
        all_imports = set()
        for entry in conv:
            if entry.get('type') == 'code_call':
                for line in entry.get('code', '').split('\n'):
                    line = line.strip()
                    if line.startswith('import ') or line.startswith('from '):
                        all_imports.add(line)
        if all_imports:
            print(f'     Libraries: {" | ".join(sorted(all_imports))}')

        # Print conversation flow: turn by turn
        if not conv:
            print(f'     (no conversation logged)')
            return

        current_turn = 0
        for entry in conv:
            turn = entry.get('turn', 0)
            etype = entry.get('type', '')

            if turn != current_turn:
                current_turn = turn
                print(f'\n     [Turn {turn}]')

            if etype == 'reasoning':
                text = entry.get('text', '')
                print(f'     [Reasoning] ({len(text)} chars):')
                for line in text.split('\n'):
                    print(f'       | {line}')

            elif etype == 'code_call':
                code = entry.get('code', '')
                output = entry.get('output', '')
                is_err = entry.get('error', False)
                err_tag = ' [ERROR]' if is_err else ''

                print(f'     [Code]{err_tag}:')
                for line in code.split('\n'):
                    print(f'       | {line}')

                if output.strip():
                    print(f'     [Output]{err_tag}:')
                    for line in output.split('\n'):
                        print(f'       > {line}')

    def _get_gpu_stats():
        """Get GPU memory stats if available."""
        try:
            import torch
            if torch.cuda.is_available():
                mem_alloc = torch.cuda.memory_allocated() / 1e9
                mem_reserved = torch.cuda.memory_reserved() / 1e9
                mem_total = torch.cuda.get_device_properties(0).total_mem / 1e9
                gpu_name = torch.cuda.get_device_name(0)
                return {
                    'name': gpu_name,
                    'allocated_gb': round(mem_alloc, 2),
                    'reserved_gb': round(mem_reserved, 2),
                    'total_gb': round(mem_total, 2),
                    'util_pct': round(100 * mem_alloc / mem_total, 1) if mem_total > 0 else 0
                }
        except Exception:
            pass
        return None

    def run_test_batch(name, problems, solver):
        """Run a batch of problems with maximum diagnostic logging."""
        solver.problems_remaining = len(problems)
        correct = 0
        results = []
        batch_start = time.time()

        print(f'\n{"="*70}')
        print(f'  {name} ({len(problems)} problems)')
        print(f'{"="*70}')

        for i, prob in enumerate(problems):
            prob_start = time.time()

            print(f'\n{"~"*70}')
            print(f'  [{i+1}/{len(problems)}] Problem id={prob["id"]}')
            print(f'{"~"*70}')

            # solve_problem prints: problem text, budget, attempt DataFrame, vote table
            predicted = solver.solve_problem(prob['problem'])

            elapsed = time.time() - prob_start
            expected = prob['answer']
            is_correct = (predicted == expected)
            if is_correct:
                correct += 1

            # Retrieve detailed attempt results
            details = getattr(solver, '_last_detailed_results', []) or []

            # --- Compute aggregate stats ---
            n_attempts = len(details)
            n_answered = sum(1 for r in details if r.get('Answer') is not None)
            n_errors = sum(r.get('Python Errors', 0) for r in details)
            n_code_calls = sum(r.get('Python Calls', 0) for r in details)
            total_tokens = sum(r.get('Response Length', 0) for r in details)
            answers = [r['Answer'] for r in details if r.get('Answer') is not None]
            answer_counts = Counter(answers)
            early_stopped = any(c >= solver.cfg.early_stop for c in answer_counts.values())
            finite_ents = [r['Entropy'] for r in details
                          if r.get('Entropy') is not None and r['Entropy'] != float('inf')]
            avg_entropy = sum(finite_ents) / len(finite_ents) if finite_ents else float('inf')
            attempt_times = [r.get('Time', 0) for r in details if r.get('Time', 0) > 0]

            # --- Result header ---
            status = 'CORRECT' if is_correct else '*** WRONG ***'
            print(f'\n  >> {status}')
            print(f'     Predicted: {predicted} | Expected: {expected} | Wall time: {elapsed:.1f}s')
            print(f'     Attempts answered: {n_answered}/{n_attempts} | '
                  f'Code calls: {n_code_calls} | Errors: {n_errors} | '
                  f'Tokens: {total_tokens}')
            ent_str = f'{avg_entropy:.3f}' if avg_entropy != float('inf') else 'inf'
            print(f'     Avg entropy: {ent_str} | '
                  f'Early stop: {"Yes" if early_stopped else "No"} (threshold={solver.cfg.early_stop})')
            if attempt_times:
                print(f'     Attempt times: min={min(attempt_times):.1f}s  max={max(attempt_times):.1f}s  '
                      f'avg={sum(attempt_times)/len(attempt_times):.1f}s')

            # GPU stats
            gpu = _get_gpu_stats()
            if gpu:
                print(f'     GPU: {gpu["name"]} | Memory: {gpu["allocated_gb"]:.1f}GB allocated / '
                      f'{gpu["total_gb"]:.1f}GB total ({gpu["util_pct"]}%)')

            # --- Vote distribution ---
            if answer_counts:
                vote_str = ' | '.join(f'{ans}: {cnt} votes' for ans, cnt in answer_counts.most_common())
                print(f'     Votes: [{vote_str}]')

            # --- Diagnosis for wrong answers ---
            if not is_correct and details:
                if predicted != expected and predicted in answer_counts:
                    pred_votes = answer_counts[predicted]
                    exp_votes = answer_counts.get(expected, 0)
                    if exp_votes > 0:
                        print(f'\n     Why wrong: {predicted} got {pred_votes} votes (chosen), '
                              f'{expected} got {exp_votes} votes (correct).')
                        print(f'     Entropy-weighted scoring preferred {predicted}.')
                    else:
                        print(f'\n     Why wrong: Correct answer {expected} was never produced.')
                        print(f'     All answers produced: {dict(answer_counts.most_common())}')

            # --- FULL DUMP of every attempt ---
            if details:
                sorted_details = sorted(details, key=lambda x: x.get('Attempt', 0))
                print(f'\n     {"="*50}')
                print(f'     FULL ATTEMPT LOGS ({n_attempts} attempts)')
                print(f'     {"="*50}')
                for r in sorted_details:
                    _print_attempt_full(r, expected)

            # --- Running totals ---
            batch_elapsed = time.time() - batch_start
            avg_time = batch_elapsed / (i + 1)
            remaining_est = avg_time * (len(problems) - i - 1)
            print(f'\n     Running: {correct}/{i+1} ({100*correct/(i+1):.1f}%) | '
                  f'Elapsed: {batch_elapsed:.0f}s | Est remaining: {remaining_est:.0f}s')

            results.append({
                'id': prob['id'],
                'predicted': predicted,
                'expected': expected,
                'correct': is_correct,
                'time_s': round(elapsed, 1),
                'answered': f'{n_answered}/{n_attempts}',
                'code_calls': n_code_calls,
                'errors': n_errors,
                'tokens': total_tokens,
                'early_stop': early_stopped,
                'votes': dict(answer_counts),
            })

        # --- Batch summary ---
        batch_total = time.time() - batch_start
        print(f'\n{"="*70}')
        print(f'  {name} COMPLETE')
        print(f'  Score: {correct}/{len(problems)} ({100*correct/len(problems):.1f}%)')
        print(f'  Total time: {batch_total:.0f}s ({batch_total/60:.1f} min) | '
              f'Avg per problem: {batch_total/len(problems):.0f}s')
        if results:
            times = [r['time_s'] for r in results]
            print(f'  Fastest: {min(times):.1f}s | Slowest: {max(times):.1f}s | '
                  f'Median: {sorted(times)[len(times)//2]:.1f}s')
        print(f'{"="*70}\n')

        return correct, len(problems), results

    def load_csv_problems(problems_csv, answers_csv):
        """Load problems from CSV files (test_fixed_50, test_random_50)."""
        import csv
        problems = {}
        with open(problems_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                problems[row['id']] = row['problem']
        answers = {}
        with open(answers_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                answers[row['id']] = int(row['answer'])
        return [{'id': pid, 'problem': problems[pid], 'answer': answers[pid]}
                for pid in problems if pid in answers]

    # =========================================================================
    #  RUN TESTS
    # =========================================================================
    all_results = []
    total_correct = 0
    total_problems = 0
    suite_start = time.time()

    # --- LEVEL 1: Reference problems (embedded, no dataset needed) ---
    if TEST_LEVEL >= 1:
        REFERENCE_PROBLEMS = [
            {'id': '92ba6a', 'problem': 'Alice and Bob are each holding some integer number of sweets. Alice says to Bob: "If we each added the number of sweets we\'re holding to our (positive integer) age, my answer would be double yours. If we took the product, then my answer would be four times yours." Bob replies: "Why don\'t you give me five of your sweets because then both our sum and product would be equal." What is the product of Alice and Bob\'s ages?', 'answer': 50},
            {'id': '9c1c5f', 'problem': 'Let $f \\colon \\mathbb{Z}_{\\geq 1} \\to \\mathbb{Z}_{\\geq 1}$ be a function such that for all positive integers $m$ and $n$, $f(m) + f(n) = f(m + n + mn)$. Across all functions $f$ such that $f(n) \\leq 1000$ for all $n \\leq 1000$, how many different values can $f(2024)$ take?', 'answer': 580},
            {'id': 'a295e9', 'problem': 'A $500 \\times 500$ square is divided into $k$ rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\\mathcal{K}$. What is the remainder when $k$ is divided by $10^{5}$?', 'answer': 520},
            {'id': '86e8e5', 'problem': 'Let $n \\geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define $g(c)=\\frac{1}{2025!}\\left\\lfloor \\frac{2025! f(M+c)}{M}\\right\\rfloor.$ We can write $g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\\frac{p}{q}$ where $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?', 'answer': 8687},
            {'id': '0e644e', 'problem': 'Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \\neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=AB$. Find the remainder when $abc$ is divided by $10^{5}$.', 'answer': 336},
            {'id': '26de63', 'problem': 'Define a function $f \\colon \\mathbb{Z}_{\\geq 1} \\to \\mathbb{Z}_{\\geq 1}$ by $f(n) = \\sum_{i = 1}^n \\sum_{j = 1}^n j^{1024} \\left\\lfloor\\frac1j + \\frac{n-i}{n}\\right\\rfloor.$ Let $M=2 \\cdot 3 \\cdot 5 \\cdot 7 \\cdot 11 \\cdot 13$ and let $N = f{\\left(M^{15}\\right)} - f{\\left(M^{15}-1\\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?', 'answer': 32951},
            {'id': '424e18', 'problem': 'A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of $20$ rounds with each runner starting with a score of $0$. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points. At the end of the tournament, we rank the competitors according to their scores. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^{5}$?', 'answer': 21818},
            {'id': '42d360', 'problem': 'On a blackboard, Ken starts off by writing a positive integer $n$ and then applies the following move until he first reaches $1$. Given that the number on the board is $m$, he chooses a base $b$, where $2 \\leq b \\leq m$, and considers the unique base-$b$ representation of $m$, $m = \\sum_{k = 0}^\\infty a_k \\cdot b^k$ where $a_k$ are non-negative integers and $0 \\leq a_k < b$ for each $k$. Ken then erases $m$ on the blackboard and replaces it with $\\sum\\limits_{k = 0}^\\infty a_k$. Across all choices of $1 \\leq n \\leq 10^{10^5}$, the largest possible number of moves Ken could make is $M$. What is the remainder when $M$ is divided by $10^{5}$?', 'answer': 32193},
            {'id': '641659', 'problem': 'Let $ABC$ be a triangle with $AB \\neq AC$, circumcircle $\\Omega$, and incircle $\\omega$. Let the contact points of $\\omega$ with $BC$, $CA$, and $AB$ be $D$, $E$, and $F$, respectively. Let the circumcircle of $AFE$ meet $\\Omega$ at $K$ and let the reflection of $K$ in $EF$ be $K\'$. Let $N$ denote the foot of the perpendicular from $D$ to $EF$. The circle tangent to line $BN$ and passing through $B$ and $K$ intersects $BC$ again at $T \\neq B$. Let sequence $(F_n)_{n \\geq 0}$ be defined by $F_0 = 0$, $F_1 = 1$ and for $n \\geq 2$, $F_n = F_{n-1} + F_{n-2}$. Call $ABC$ $n$-tastic if $BD = F_n$, $CD = F_{n+1}$, and $KNK\'B$ is cyclic. Across all $n$-tastic triangles, let $a_n$ denote the maximum possible value of $\\frac{CT \\cdot NB}{BT \\cdot NE}$. Let $\\alpha$ denote the smallest real number such that for all sufficiently large $n$, $a_{2n} < \\alpha$. Given that $\\alpha = p + \\sqrt{q}$ for rationals $p$ and $q$, what is the remainder when $\\left\\lfloor p^{q^p} \\right\\rfloor$ is divided by $99991$?', 'answer': 57447},
            {'id': 'dd7f5e', 'problem': 'Let $\\mathcal{F}$ be the set of functions $\\alpha \\colon \\mathbb{Z}\\to \\mathbb{Z}$ for which there are only finitely many $n \\in \\mathbb{Z}$ such that $\\alpha(n) \\neq 0$. For two functions $\\alpha$ and $\\beta$ in $\\mathcal{F}$, define their product $\\alpha\\star\\beta$ to be $\\sum\\limits_{n\\in\\mathbb{Z}} \\alpha(n)\\cdot \\beta(n)$. Also, for $n\\in\\mathbb{Z}$, define a shift operator $S_n \\colon \\mathcal{F}\\to \\mathcal{F}$ by $S_n(\\alpha)(t)=\\alpha(t+n)$ for all $t \\in \\mathbb{Z}$. A function $\\alpha \\in \\mathcal{F}$ is called shifty if $\\alpha(m)=0$ for all integers $m<0$ and $m>8$ and there exists $\\beta \\in \\mathcal{F}$ and integers $k \\neq l$ such that for all $n \\in \\mathbb{Z}$, $S_n(\\alpha)\\star\\beta = 1$ if $n \\in \\{k,l\\}$ and $0$ if $n \\not\\in \\{k,l\\}$. How many shifty functions are there in $\\mathcal{F}$?', 'answer': 160},
        ]
        c, t, r = run_test_batch('REFERENCE PROBLEMS', REFERENCE_PROBLEMS, solver)
        total_correct += c
        total_problems += t
        all_results.extend(r)

    # --- LEVEL 2: IMO-Level Hard Benchmark (from hard_benchmark_30.csv) ---
    # 28 IMO-AnswerBench problems, 7 per domain (combinatorics, geometry, number_theory, algebra)
    # Replaces hand-picked diagnostic set with harder, more diverse problems
    if TEST_LEVEL >= 2:
        hard_bench_csv = '/kaggle/input/aimo3-test-data/hard_benchmark_30.csv'
        if os.path.isfile(hard_bench_csv):
            import csv
            hard_problems = []
            with open(hard_bench_csv, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    hard_problems.append({
                        'id': row['id'],
                        'problem': row['problem'],
                        'answer': int(row['answer'])
                    })
            # Take first 10 for Level 2 (mix of domains)
            import random
            random.seed(2026)
            random.shuffle(hard_problems)
            hard_10 = hard_problems[:10]
            c, t, r = run_test_batch('HARD BENCHMARK (10 IMO-level)', hard_10, solver)
            total_correct += c
            total_problems += t
            all_results.extend(r)
        else:
            # Fallback to old fixed 10 from test_fixed_50
            fixed_csv = '/kaggle/input/aimo3-test-data/test_fixed_50.csv'
            fixed_ans = '/kaggle/input/aimo3-test-data/test_fixed_50_answers.csv'
            FIXED_10_IDS = [
                'f8b3d5', 'c854cd', '42ee2a', '269012', '35ecd8',
                'c8adf3', '6837b1', '83ce64', '6618d2', 'f9333c',
            ]
            if os.path.isfile(fixed_csv) and os.path.isfile(fixed_ans):
                all_fixed = load_csv_problems(fixed_csv, fixed_ans)
                fixed_by_id = {p['id']: p for p in all_fixed}
                fixed_problems = [fixed_by_id[pid] for pid in FIXED_10_IDS if pid in fixed_by_id]
                c, t, r = run_test_batch('FIXED 10 DIAGNOSTIC (fallback)', fixed_problems, solver)
                total_correct += c
                total_problems += t
                all_results.extend(r)
            else:
                print(f'SKIPPING Level 2: no hard benchmark or fixed dataset found')

    # --- LEVEL 3: 10 Random Problems (from remaining pool) ---
    if TEST_LEVEL >= 3:
        fixed_csv = '/kaggle/input/aimo3-test-data/test_fixed_50.csv'
        fixed_ans = '/kaggle/input/aimo3-test-data/test_fixed_50_answers.csv'
        if os.path.isfile(fixed_csv) and os.path.isfile(fixed_ans):
            all_pool = load_csv_problems(fixed_csv, fixed_ans)
            tested_ids = {r['id'] for r in all_results}
            remaining = [p for p in all_pool if p['id'] not in tested_ids]
            import random
            random.seed(2026)
            random.shuffle(remaining)
            rand_problems = remaining[:10]
            if rand_problems:
                c, t, r = run_test_batch('RANDOM 10 (from remaining pool)', rand_problems, solver)
                total_correct += c
                total_problems += t
                all_results.extend(r)
            else:
                print('SKIPPING random 10: no remaining problems after dedup')
        else:
            print(f'SKIPPING random 10: dataset not found at {fixed_csv}')

    # --- LEVEL 4: 30 Fixed Comprehensive Benchmark ---
    if TEST_LEVEL >= 4:
        fixed_csv = '/kaggle/input/aimo3-test-data/test_fixed_50.csv'
        fixed_ans = '/kaggle/input/aimo3-test-data/test_fixed_50_answers.csv'
        if os.path.isfile(fixed_csv) and os.path.isfile(fixed_ans):
            all_pool = load_csv_problems(fixed_csv, fixed_ans)
            tested_ids = {r['id'] for r in all_results}
            remaining = [p for p in all_pool if p['id'] not in tested_ids]
            if remaining:
                c, t, r = run_test_batch(f'COMPREHENSIVE BENCHMARK ({len(remaining)} problems)', remaining, solver)
                total_correct += c
                total_problems += t
                all_results.extend(r)
            else:
                print('SKIPPING comprehensive: all problems already tested')
        else:
            print(f'SKIPPING comprehensive: dataset not found at {fixed_csv}')


    # =========================================================================
    #  DOUBLE-RUN RETRY: Re-run failed problems to simulate competition scoring
    # =========================================================================
    wrong = [r for r in all_results if not r['correct']]
    if wrong:
        print(f'\n{"="*70}')
        print(f'  DOUBLE-RUN RETRY: Re-solving {len(wrong)} failed problems')
        print(f'{"="*70}\n')
        
        retry_problems = []
        for r in wrong:
            # Find original problem data from all tested problems
            for src_list in [ref_problems if 'ref_problems' in dir() else [],
                             fixed_problems if 'fixed_problems' in dir() else [],
                             rand_problems if 'rand_problems' in dir() else [],
                             comp_problems if 'comp_problems' in dir() else []]:
                for p in src_list:
                    if p['id'][:6] == r['id']:
                        retry_problems.append(p)
                        break
        
        if retry_problems:
            retry_c, retry_t, retry_results = run_test_batch(
                'DOUBLE-RUN RETRY', retry_problems, solver)
            
            # Compute double-run scoring
            print(f'\n{"="*70}')
            print(f'  DOUBLE-RUN SCORE SIMULATION')
            print(f'{"="*70}')
            
            # Build lookup: problem_id -> (run1_correct, run2_correct)
            run1_by_id = {r['id']: r['correct'] for r in all_results}
            run2_by_id = {r['id']: r['correct'] for r in retry_results}
            
            double_score = 0
            problems_with_double = 0
            for pid in run1_by_id:
                r1 = run1_by_id[pid]
                if pid in run2_by_id:
                    # This problem was retried
                    r2 = run2_by_id[pid]
                    if r1 and r2:
                        pts = 1.0
                    elif r1 or r2:
                        pts = 0.5
                    else:
                        pts = 0.0
                    problems_with_double += 1
                    print(f'  {pid}: Run1={r1} Run2={r2} -> {pts} pts')
                else:
                    # Only ran once — assume both runs same (best case)
                    pts = 1.0 if r1 else 0.0
                double_score += pts
            
            print(f'\n  Double-run estimated score: {double_score}/{total_problems}')
            print(f'  Single-run score: {total_correct}/{total_problems}')
            print(f'  Delta: {double_score - total_correct:+.1f} (negative = double-run penalty)')
    
    # =========================================================================
    #  FINAL SUMMARY
    # =========================================================================
    suite_total = time.time() - suite_start
    if all_results:
        print(f'\n{"="*70}')
        print(f'  FINAL SUMMARY')
        print(f'  Score: {total_correct}/{total_problems} ({100*total_correct/total_problems:.1f}%)')
        print(f'  Total suite time: {suite_total:.0f}s ({suite_total/60:.1f} min)')
        print(f'{"="*70}\n')

        summary_df = pd.DataFrame([
            {k: v for k, v in r.items() if k != 'votes'}
            for r in all_results
        ])
        display(summary_df)

        # Wrong answers breakdown
        wrong = [r for r in all_results if not r['correct']]
        if wrong:
            print(f'\nWRONG ANSWERS ({len(wrong)}):')
            for r in wrong:
                print(f'  {r["id"]}: predicted={r["predicted"]} expected={r["expected"]} '
                      f'time={r["time_s"]}s votes={r["votes"]}')

        # Correct answers summary
        correct_list = [r for r in all_results if r['correct']]
        if correct_list:
            avg_correct_time = sum(r['time_s'] for r in correct_list) / len(correct_list)
            print(f'\nCORRECT ({len(correct_list)}): avg time={avg_correct_time:.1f}s')